# Debugging Rotated QR Codes

This notebook investigates why rotated QR codes (30° and 45°) are failing to decode, even after ML detection.

We will:
1. Load a failing image (`generated_dataset/text_rot_45.png`).
2. Visualize the rotation and artifacts.
3. Try decoding with `pyzbar` (ZBar) and `cv2` (WeChatQr) to see if other libraries can handle it.
4. Experiment with preprocessing (thresholding, sharpening).

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pyzbar.pyzbar import decode

# Helper to display images
def show(img, title="Image"):
    plt.figure(figsize=(6, 6))
    if len(img.shape) == 2:
        plt.imshow(img, cmap='gray')
    else:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis('off')
    plt.show()

# Load failing image
img_path = "generated_dataset/text_rot_45.png"
img = cv2.imread(img_path)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

show(img, "Original Image")

In [ ]:
# Try decoding with pyzbar (ZBar)
decoded = decode(img)
print("ZBar Results:", decoded)

# Try with OpenCV WeChatQRCode (if available)
try:
    detector = cv2.wechat_qrcode_WeChatQRCode()
    res, points = detector.detectAndDecode(img)
    print("WeChatQR Results:", res)
except Exception as e:
    print("WeChatQR not available or failed:", e)

In [ ]:
# Simulate the Rust pipeline's rotation
def rotate_image(image, angle):
    image_center = tuple(np.array(image.shape[1::-1]) / 2)
    rot_mat = cv2.getRotationMatrix2D(image_center, angle, 1.0)
    result = cv2.warpAffine(image, rot_mat, image.shape[1::-1], flags=cv2.INTER_LINEAR)
    return result

# Rotate -45 degrees (to make it upright)
rotated = rotate_image(gray, -45)
show(rotated, "Rotated -45 (Upright?)")

# Try decoding the rotated image with ZBar
decoded_rot = decode(rotated)
print("ZBar Results on Rotated:", decoded_rot)